# The LLM Gateway - Portkey


---

### The problem, in one picture

Without a gateway, every app talks to every provider directly:

```
                    ┌──────────► OpenAI      (openai SDK,   key #1)
   Your app ────────┼──────────► Anthropic   (anthropic SDK, key #2)
                    └──────────► Gemini      (google SDK,   key #3)
```

So you end up writing:

- 3 different SDKs, 3 different response shapes
- 3 API keys pasted in 3 places
- No single place to see cost, latency, or errors
- No retry. If OpenAI hiccups, your app is down


With a gateway, there is **one door**:

```
                         ┌─────────────┐  ┌──────► OpenAI
   Your app ────────────►│   PORTKEY   │──┼──────► Anthropic
      (one SDK)          │   GATEWAY   │  └──────► Gemini
                         └──────┬──────┘
                                │ and for free you get:
                                ├─   logs, cost, latency
                                ├─   retry + fallback
                                ├─   load balancing
                                └─   caching
```

> ## **A gateway is a smart middleman between your app and the model providers.**

# What is Portkey, and why use it?

### The 5 jobs a gateway does for you

| Job | Without a gateway | With Portkey |
|---|---|---|
| **One API** | A different SDK per provider | One SDK, swap models by changing a string |
| **Observability** | You have no idea what it costs | Every call logged: cost, tokens, latency |
| **Reliability** | One provider is down = you are down | Retry, then fall back to another model |
| **Load balancing** | All traffic to one place | Split traffic by weight |
| **Caching** | Pay twice for the same question | Repeat answers are free and instant |


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()


PROVIDER = os.environ.get("PORTKEY_PROVIDER", "@openai-prod")
MODEL = os.environ.get("PORTKEY_MODEL", "gpt-4o-mini")

if os.environ.get("PORTKEY_API_KEY"):
    print("PORTKEY_API_KEY loaded")
else:
    print("PORTKEY_API_KEY MISSING, add it to your .env file")

print("Provider:", PROVIDER)
print("Model   :", MODEL)

PORTKEY_API_KEY loaded
Provider: @openai-prod
Model   : gpt-4o-mini


In [2]:
from portkey_ai import Portkey

client = Portkey(
    api_key = os.environ["PORTKEY_API_KEY"],
    Provider = PROVIDER,
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in exactly 5 words."}],
)

In [3]:
print(response.choices[0].message.content)

Hello! How are you today?


In [4]:
print("answer :", response.choices[0].message.content)
print("model  :", response.model)
print("tokens :", response.usage.total_tokens)

answer : Hello! How are you today?
model  : gpt-4o-mini-2024-07-18
tokens : 22


In [5]:
def ask(client, question):
    """Send one question through the gateway and return the response object."""
    return client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
    )

response = ask(client, "What is an API gateway? One short sentence.")
print(response.choices[0].message.content)

An API gateway is a server that acts as a single entry point for managing, routing, and securing API requests between clients and backend services.


User Tracing

### The problem

10,000 requests are in your logs. A customer emails: *"your bot gave me a wrong answer
this morning."* Which of the 10,000 was theirs?

### The fix: label every request

Portkey gives you two labels:

| Label | What it is | Use it for |
|---|---|---|
| **`trace_id`** | Your own ID for **one user action** | Grouping the 3 calls that made up one answer |
| **`metadata`** | Any key/value labels you like | Filtering by user, feature, environment |

**`_user` is a special metadata key.** Portkey uses it to build per-user analytics
(cost per user, requests per user), so always set it.

In [6]:
response = client.with_options(
    trace_id = "chat-001",
    metadata = {"_user": "thiru@example.com", "feature": "support-bot"},).chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "How do I reset my password?"}],
    )

print(response.choices[0].message.content)

To reset your password, you typically need to follow these general steps, which can vary depending on the platform or service you are using:

1. **Visit the Login Page**: Go to the website or application where you need to reset your password.

2. **Click on 'Forgot Password?'**: Look for a link or button that says “Forgot Password?” or “Reset Password.” This is usually located near the password entry field.

3. **Enter Your Information**: You may be asked to enter your email address, username, or phone number associated with your account.

4. **Check Your Email or SMS**: After submitting your information, check your email inbox or SMS for a password reset link or code. Make sure to check your spam or junk folder as well.

5. **Follow the Instructions**: Click on the link or enter the code you received, and follow the instructions provided to create a new password.

6. **Create a Strong Password**: Choose a strong and unique password that you haven’t used before.

7. **Log In**: Once yo

---
# Retry, Timeout and Fallback

### Everything from here is one idea: the **config**

A **config** is a small dictionary that tells the gateway how to behave.
You hand it to the client, and the gateway does the work. **Your calling code never changes.**

```python
client = Portkey(api_key=..., config={ ... the rules ... })
```

That is the whole mental model. Now let's write four of them.

### 4.1 - Retry

LLM APIs return **429 (rate limited)** and **500 (server error)** all the time.
Without a gateway you write try/except loops. With one, you write this:

In [13]:
retry_config = {
    "retry": {
        "attempts": 3,                            
        "on_status_codes": [429, 500, 502, 503],  
    }
}

retry_client = Portkey(
    api_key = os.environ["PORTKEY_API_KEY"],
    Provider = PROVIDER,
    config = "pc-gatewa-b557be",
)

response = ask(retry_client, "Say OK.")

print(response.choices[0].message.content)
print("retries used:", response.get_headers().get("retry-attempt-count"))

OK.
retries used: 0


 Timeout ⏱

A model that hangs for 60 seconds is worse than one that fails fast.
Set a deadline and move on.

> **The unit is milliseconds**, not seconds. `request_timeout: 10000` is 10 seconds.

In [14]:
timeout_config = {
        "request_timeout": 10000,
}

retry_client = Portkey(
    api_key = os.environ["PORTKEY_API_KEY"],
    Provider = PROVIDER,
    config = "pc-gatewa-6e1b4a",
)

response = ask(retry_client, "Say OK.")

print(response.choices[0].message.content)

OK.


In [15]:
impatient_config = {
    "request_timeout": 1        
}

impatient_client = Portkey(
    api_key=os.environ["PORTKEY_API_KEY"],
    provider=PROVIDER,
    config = "pc-test-t-4762e6",
)

try:
    response = ask(impatient_client, "Say OK.")
    print(response.choices[0].message.content)
except Exception as error:
    print("Timed out, as expected.")
    print("error:", str(error)[:120])

Timed out, as expected.
error: Error code: 408 - {'error': {'message': 'Request exceeded the timeout sent in the request: 1ms', 'type': 'timeout_error'


> **Why this matters:** a timeout is a **408**, and a 408 can trigger a fallback.
> So "wait 10 seconds, then try a different model" is just two config keys together.

### 4.3 - Fallback

**If the first model fails, automatically try the second one.** No code change.

Here the first target is a model name that does not exist, so it will fail and the
gateway will quietly use the second one.

In [16]:
fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": PROVIDER + "/this-model-does-not-exist"}},
        {"override_params": {"model": PROVIDER + "/" + MODEL}},
    ],
}

fallback_client = Portkey(
    api_key=os.environ["PORTKEY_API_KEY"],
    config="pc-gatewa-bf51cf",
)

response = fallback_client.chat.completions.create(
    messages=[{"role": "user", "content": "Say OK."}],
)

print("answer      :", response.choices[0].message.content)
print("model used  :", response.model)
print("target index:", response.get_headers().get("last-used-option-index"))

answer      : OK.
model used  : gpt-4o-mini-2024-07-18
target index: config.targets[0]


---
# Part 5 - Load balancing and Caching

### 5.1 - Load balancing

**Split traffic between models by weight.**

Why you would want this:

- Send 80% of traffic to the cheap model, 20% to the smart one
- A/B test two models on real users
- Spread load across two accounts to avoid rate limits

Weights are **relative**, so they do not need to add up to 1.

In [17]:
loadbalance_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {"override_params": {"model": PROVIDER + "/gpt-4o-mini"}, "weight": 0.8},
        {"override_params": {"model": PROVIDER + "/gpt-4o"},      "weight": 0.2},
    ],
}

balanced_client = Portkey(
    api_key=os.environ["PORTKEY_API_KEY"],
    config="pc-gatewa-7a1aba",
)

for i in range(6):
    response = balanced_client.chat.completions.create(
        messages=[{"role": "user", "content": "Say OK."}],
    )
    print(i + 1, "->", response.model)

1 -> gpt-4o-mini-2024-07-18
2 -> gpt-4o-mini-2024-07-18
3 -> gpt-4o-mini-2024-07-18
4 -> gpt-4o-mini-2024-07-18
5 -> gpt-4o-2024-08-06
6 -> gpt-4o-mini-2024-07-18




### 5.2 - Caching

**If the same question comes in twice, do not pay for it twice.**

| Mode | Matches when | Use for |
|---|---|---|
| `"simple"` | The request is **exactly** the same | FAQs, fixed prompts, testing |
| `"semantic"` | The request **means** the same thing | Real user chat, worded differently |

> **`max_age` is in SECONDS** (unlike `request_timeout`, which is milliseconds).
> Minimum 60, maximum 7776000 (90 days), default 604800 (7 days).

In [19]:
cache_config = {
    "cache": {
        "mode": "simple",
        "max_age": 300,     
    }
}

cached_client = Portkey(
    api_key=os.environ["PORTKEY_API_KEY"],
    provider=PROVIDER,
    config="pc-gatewa-50330a",
)

print("cache is on")

cache is on


In [20]:
import time

question = "Name the seven colours of the rainbow."

for attempt in [1, 2]:
    start = time.time()

    response = ask(cached_client, question)

    seconds = round(time.time() - start, 2)
    status = response.get_headers().get("cache-status")

    print("call", attempt, "took", seconds, "seconds | cache:", status)

call 1 took 5.03 seconds | cache: MISS
call 2 took 1.33 seconds | cache: MISS
